# LatAm Oil Field Valuation: Fiscal Regime & DCF Model

**Research Question:** How do differences in fiscal regimes across Latin American oil-producing countries affect the NPV of an identical upstream asset?

**Author:** Victor Hugo Campos Reis Alves | M.Sc. Economics, UFPE/PIMES  
**Contact:** victor.camposr@outlook.com | [GitHub](https://github.com/victorcampos-reis23)

---

## 1. Motivation

Two identical oil fields can produce dramatically different investor returns depending on the country fiscal regime. This model quantifies that difference across Brazil, Mexico, Colombia, and Argentina using real fiscal legislation.

| Country | Regime | Key Instruments |
|---|---|---|
| Brazil | Concession | ANP Royalties + Participacao Especial + CSLL/IRPJ |
| Mexico | License/PSC | CNH Royalties (price-indexed) + ISR |
| Colombia | Concession | ANH Royalties (sliding) + Income Tax |
| Argentina | Concession | Royalties + Export Tax (retenciones) + Income Tax |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../outputs', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

COLORS = {
    'Brazil':    '#009c3b',
    'Mexico':    '#ce1126',
    'Colombia':  '#b8860b',
    'Argentina': '#4a90d9',
    'neutral':   '#2c3e50',
}

print('Libraries loaded.')

## 2. Asset Parameters

In [ ]:
ASSET = {
    'reserves_mmbbl':   100,
    'peak_prod_mbopd':  20,
    'ramp_years':       3,
    'plateau_years':    5,
    'decline_rate':     0.15,
    'project_life':     20,
    'capex_usd_mm':     800,
    'capex_years':      4,
    'opex_bbl':         12,
    'abandonment_mm':   50,
    'oil_price':        75,
    'discount_rate':    0.10,
    'inflation':        0.025,
}

for k, v in ASSET.items():
    print(f'  {k:25s}: {v}')

## 3. Production Profile

In [ ]:
def build_production_profile(asset):
    years = np.arange(1, asset['project_life'] + 1)
    prod  = np.zeros(len(years))
    peak  = asset['peak_prod_mbopd'] * 365
    for i, yr in enumerate(years):
        if yr <= asset['ramp_years']:
            prod[i] = peak * (yr / asset['ramp_years'])
        elif yr <= asset['ramp_years'] + asset['plateau_years']:
            prod[i] = peak
        else:
            d = yr - asset['ramp_years'] - asset['plateau_years']
            prod[i] = peak * np.exp(-asset['decline_rate'] * d)
    return pd.DataFrame({'year': years, 'production_mbbl': prod})

prod_df = build_production_profile(ASSET)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(prod_df['year'], prod_df['production_mbbl'] / 1000,
                alpha=0.3, color=COLORS['neutral'])
ax.plot(prod_df['year'], prod_df['production_mbbl'] / 1000,
        color=COLORS['neutral'], lw=2)
ax.axvline(ASSET['ramp_years'], color='gray', ls='--', lw=0.8)
ax.axvline(ASSET['ramp_years'] + ASSET['plateau_years'], color='gray', ls='--', lw=0.8)
ax.set_title('Standardized Production Profile - 100 MMbbl Field', fontsize=12, fontweight='bold')
ax.set_xlabel('Project Year')
ax.set_ylabel('Production (MMbbl/year)')
plt.tight_layout()
plt.savefig('../outputs/fig1_production_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total production: {prod_df["production_mbbl"].sum()/1000:.1f} MMbbl')

## 4. Fiscal Regime Functions

Sources: Lei 9.478/1997 (BR), Ley de Hidrocarburos 2014 (MX), ANH Reglamento 2012 (CO), Ley 17.319 (AR).

In [ ]:
def royalty_brazil(prod_bopd):
    """ANP sliding scale: 5-15% based on daily production."""
    if prod_bopd < 5000:    return 0.05
    elif prod_bopd < 10000: return 0.08
    elif prod_bopd < 15000: return 0.10
    elif prod_bopd < 20000: return 0.12
    else:                   return 0.15

def participacao_especial(gross_rev, cum_prod_mmbbl):
    """Progressive surcharge on net revenue: 10-40% after production thresholds.
    Note: Exists within the concession regime (not production sharing).
    In the pre-salt PSC, the equivalent instrument is the profit-oil split."""
    if cum_prod_mmbbl < 5:     return 0.0
    elif cum_prod_mmbbl < 15:  return 0.10 * gross_rev
    elif cum_prod_mmbbl < 50:  return 0.20 * gross_rev
    elif cum_prod_mmbbl < 100: return 0.30 * gross_rev
    else:                      return 0.40 * gross_rev

def royalty_mexico(price):
    """CNH price-indexed royalty: 7.5-27.5%."""
    if price < 48:   return 0.075
    elif price < 60: return 0.125
    elif price < 75: return 0.175
    elif price < 90: return 0.225
    else:            return 0.275

def royalty_colombia(prod_bopd):
    """ANH sliding scale: 8-25%."""
    if prod_bopd < 5000:
        return 0.08
    else:
        return min(0.08 + (prod_bopd - 5000) * (0.17 / 120000), 0.25)

print('Fiscal regime functions ready.')

## 5. DCF Model

In [ ]:
def run_dcf(asset, country, prod_df):
    results  = []
    price    = asset['oil_price']
    cum_prod = 0
    capex_schedule = np.zeros(asset['project_life'])
    capex_schedule[:asset['capex_years']] = asset['capex_usd_mm'] / asset['capex_years']

    for i, row in prod_df.iterrows():
        yr        = int(row['year'])
        prod_mbbl = row['production_mbbl']
        prod_bopd = prod_mbbl * 1000 / 365
        cum_prod += prod_mbbl / 1000

        opex_bbl   = asset['opex_bbl'] * (1 + asset['inflation']) ** (yr - 1)
        gross_rev  = prod_mbbl * price
        opex_total = prod_mbbl * opex_bbl
        capex      = capex_schedule[i]
        abandon    = asset['abandonment_mm'] if yr == asset['project_life'] else 0

        if country == 'Brazil':
            royalty    = gross_rev * royalty_brazil(prod_bopd)
            part_esp   = participacao_especial(gross_rev, cum_prod)
            tax        = max(0, (gross_rev - royalty - opex_total - capex) * 0.34)
            total_govt = royalty + part_esp + tax

        elif country == 'Mexico':
            royalty    = gross_rev * royalty_mexico(price)
            tax        = max(0, (gross_rev - royalty - opex_total - capex) * 0.30)
            total_govt = royalty + tax

        elif country == 'Colombia':
            royalty    = gross_rev * royalty_colombia(prod_bopd)
            tax        = max(0, (gross_rev - royalty - opex_total - capex) * 0.35)
            total_govt = royalty + tax

        elif country == 'Argentina':
            royalty    = gross_rev * 0.15
            export_tax = gross_rev * 0.08
            tax        = max(0, (gross_rev - royalty - export_tax - opex_total - capex) * 0.35)
            total_govt = royalty + export_tax + tax

        fcf = gross_rev - total_govt - opex_total - capex - abandon
        pv  = fcf / (1 + asset['discount_rate']) ** yr

        results.append({
            'year': yr, 'gross_rev': gross_rev,
            'total_govt': total_govt, 'opex': opex_total,
            'capex': capex, 'fcf': fcf, 'pv_fcf': pv,
            'govt_take_pct': total_govt / gross_rev * 100 if gross_rev > 0 else 0,
        })

    return pd.DataFrame(results)

countries   = ['Brazil', 'Mexico', 'Colombia', 'Argentina']
dcf_results = {c: run_dcf(ASSET, c, prod_df) for c in countries}

print('DCF Results - NPV by Country (USD million):')
print('-' * 50)
for c, df in dcf_results.items():
    npv  = df['pv_fcf'].sum()
    take = df['govt_take_pct'].mean()
    print(f'{c:12s}  NPV: ${npv:8.1f}M  |  Avg Govt Take: {take:.1f}%')

## 6. Visualizations

In [ ]:
# Figure 2: NPV Comparison
npvs     = {c: dcf_results[c]['pv_fcf'].sum() for c in countries}
sorted_c = sorted(npvs, key=npvs.get, reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sorted_c, [npvs[c] for c in sorted_c],
               color=[COLORS[c] for c in sorted_c], edgecolor='white', height=0.5)
for bar, c in zip(bars, sorted_c):
    ax.text(bar.get_width() * 1.03, bar.get_y() + bar.get_height()/2,
            f'${npvs[c]:.0f}M', va='center', fontsize=11, fontweight='bold')
ax.set_title('NPV Comparison: Identical Asset Under LatAm Fiscal Regimes\n'
             '(100 MMbbl | $75/bbl | 10% WACC)', fontsize=12, fontweight='bold')
ax.set_xlabel('Net Present Value (USD million)')
ax.axvline(0, color='gray', lw=0.8)
plt.tight_layout()
plt.savefig('../outputs/fig2_npv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Cash Flow Waterfall
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()
for ax, country in zip(axes, countries):
    df    = dcf_results[country]
    color = COLORS[country]
    ax.fill_between(df['year'], df['gross_rev'],   alpha=0.15, color=color, label='Gross Revenue')
    ax.fill_between(df['year'], df['fcf'],          alpha=0.40, color=color, label='Free Cash Flow')
    ax.fill_between(df['year'], -df['total_govt'],  alpha=0.20, color='red',  label='Govt Take')
    ax.fill_between(df['year'], -df['capex'],       alpha=0.30, color='gray', label='CAPEX')
    ax.plot(df['year'], df['fcf'], color=color, lw=2)
    ax.axhline(0, color='black', lw=0.5)
    npv = df['pv_fcf'].sum()
    ax.set_title(f'{country} | NPV: ${npv:.0f}M', fontsize=11, fontweight='bold')
    ax.set_xlabel('Project Year')
    ax.set_ylabel('USD Million')
    if country == 'Brazil':
        ax.legend(fontsize=8)
plt.suptitle('Cash Flow Breakdown by Country - Fiscal Regime Impact', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/fig3_cashflow_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Government Take Over Time
fig, ax = plt.subplots(figsize=(12, 5))
for country in countries:
    df = dcf_results[country]
    ax.plot(df['year'], df['govt_take_pct'],
            color=COLORS[country], lw=2, label=country, marker='o', markersize=3)
ax.set_title('Government Take (%) Over Project Life', fontsize=12, fontweight='bold')
ax.set_xlabel('Project Year')
ax.set_ylabel('Government Take (% of Gross Revenue)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/fig4_govt_take.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Monte Carlo Simulation

In [ ]:
np.random.seed(42)
N_SIMS          = 5000
price_scenarios = np.clip(np.random.normal(75, 15, N_SIMS), 20, 150)

npv_mc = {c: [] for c in countries}
for price_sim in price_scenarios:
    asset_sim = ASSET.copy()
    asset_sim['oil_price'] = price_sim
    for country in countries:
        df_sim = run_dcf(asset_sim, country, prod_df)
        npv_mc[country].append(df_sim['pv_fcf'].sum())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for ax, country in zip(axes, countries):
    s   = np.array(npv_mc[country])
    p10 = np.percentile(s, 10)
    p50 = np.percentile(s, 50)
    p90 = np.percentile(s, 90)
    ax.hist(s, bins=60, color=COLORS[country], alpha=0.7, edgecolor='white')
    ax.axvline(p10, color='red',   ls='--', lw=1.5, label=f'P10: ${p10:.0f}M')
    ax.axvline(p50, color='black', ls='-',  lw=2,   label=f'P50: ${p50:.0f}M')
    ax.axvline(p90, color='green', ls='--', lw=1.5, label=f'P90: ${p90:.0f}M')
    ax.axvline(0,   color='gray',  ls=':',  lw=1)
    prob = (s > 0).mean() * 100
    ax.set_title(f'{country} | P(NPV>0): {prob:.0f}%', fontsize=11, fontweight='bold')
    ax.set_xlabel('NPV (USD million)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
plt.suptitle('Monte Carlo NPV Distribution by Fiscal Regime\n'
             '(5,000 simulations | Oil price: Normal($75,$15))',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/fig5_monte_carlo_npv.png', dpi=150, bbox_inches='tight')
plt.show()

print('P10 / P50 / P90 Results:')
print('-' * 65)
for country in countries:
    s = np.array(npv_mc[country])
    print(f'{country:12s}  '
          f'P10:${np.percentile(s,10):7.0f}M  '
          f'P50:${np.percentile(s,50):7.0f}M  '
          f'P90:${np.percentile(s,90):7.0f}M  '
          f'P(>0):{(s>0).mean()*100:.0f}%')

## 8. Summary Table

In [ ]:
rows = []
for c in countries:
    df = dcf_results[c]
    s  = np.array(npv_mc[c])
    rows.append({
        'Country':       c,
        'NPV@$75':       f"${df['pv_fcf'].sum():.0f}M",
        'Avg Govt Take': f"{df['govt_take_pct'].mean():.1f}%",
        'P10':           f"${np.percentile(s,10):.0f}M",
        'P50':           f"${np.percentile(s,50):.0f}M",
        'P90':           f"${np.percentile(s,90):.0f}M",
        'P(NPV>0)':      f"{(s>0).mean()*100:.0f}%",
    })
summary = pd.DataFrame(rows).set_index('Country')
summary.to_csv('../outputs/latam_fiscal_comparison.csv')
print(summary.to_string())